# 30 m Grid Floor + Transparent Cubes (empty.umap)

30 m グリッド床 + 透明箱（最大 10,000）+ Humanoid + SpotDog を Unreal にスポーンします。

ロジック本体は同ディレクトリの `grid_env_hri_simulation.py` です。この Notebook からはモジュールを import して段階実行します。

## 前提

1. **Windows**: `C:\SimWorldServer` で SimWorld を起動
   ```text
   .\SimWorld.exe -windowed -log /Game/Maps/empty.umap
   ```
2. **pakchunk9002** に `BP_Floor_30x30` / `BP_TransparentCube` が含まれていること
3. **WSL**: `conda activate simworld` のカーネルで実行

## 座標系（UE cm、empty マップ）

- 床 30 m 四方の **左下隅** が `(x, y) = (0, 0)`
- 床上面の高さ = **100 cm**（empty 原点から 1 m）
- Human / Robot のマップ座標 [m] は material_transport の grid_map と同じ比率（既定: human `(1,1)`, robot `(1,3)`）
- Humanoid / Robot は **床面上に直接配置**（Simulated Physics は既定 OFF。ON にするとラグドール化します）
- **透明箱（格子）**: 床 **左下角** `(0,0)` から `GRID_N×0.3 m` に `BP_TransparentCube`。**既定**は床上静置・**透過モード**（`GRID_CUBE_BLOCKING=0` → `SetBlocking False`、物理 OFF）。実体格子＋落下は `GRID_CUBE_BLOCKING=1` と `CUBE_ENABLE_PHYSICS=1`
- **デモ用 TransparentCube（pakchunk9002）**: L 字 3 個は **当たりあり**（`SetBlocking True`、見た目は半透明グレーのまま）、`(5.5,5.5)` 1 個は **通過可**（`SetBlocking False`）。SpotDog を L 字側へ動かすと当たり差を確認しやすい

## 実行順

1. 初期設定 → … → 8. デモ用 TransparentCube → 9. エージェント → 10. 落下待ち → **11. SpotDog 通過試験** → 12. クリーンアップ  
小規模テストは **設定** セルで `GRID_N = 3`（9 箱）にしてからスポーンしてください。

接続は `grid_env_hri_simulation.ensure_connection()` を使います。各候補 IP を **数秒の TCP プローブ**で確認してから接続するため、到達不能な `10.103.0.1` で数分ハングしません。WSL で `127.0.0.1:9000` が **python** のときはスキップ（`malformat magic 255` 防止）。SimWorld 起動後に **設定セルで `importlib.reload(geh)`** してから UE 接続してください。

In [ ]:
import importlib
import os
import sys
import time
from pathlib import Path
from typing import Optional, Tuple


def _find_project_root() -> Path:
    search_starts = []
    if "__file__" in globals():
        search_starts.append(Path(__file__).resolve().parent)
    search_starts.append(Path.cwd().resolve())

    for start in search_starts:
        for candidate in (start, *start.parents):
            if (candidate / "setup.py").exists() and (candidate / "simworld").is_dir():
                return candidate

    return Path.cwd().resolve().parent.parent


_root = _find_project_root()
_geh_dir = _root / "dev" / "grid_env_hri"
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
if str(_geh_dir) not in sys.path:
    sys.path.insert(0, str(_geh_dir))

from simworld.communicator.communicator import Communicator
from simworld.communicator.unrealcv import UnrealCV

ucv: Optional[UnrealCV] = None
communicator: Optional[Communicator] = None
UE_PORT = 9000

print(f"[Paths] root={_root}")
print(f"[Paths] grid_env_hri={_geh_dir}")

In [ ]:
import grid_env_hri_simulation as geh


def ensure_connection() -> Tuple[UnrealCV, Communicator]:
    """既存接続を再利用しつつ、接続ロジックは grid_env_hri_simulation.py に委譲。"""
    global ucv, communicator

    if ucv is not None and ucv.client.isconnected():
        if communicator is None or communicator.unrealcv is not ucv:
            communicator = Communicator(ucv)
        return ucv, communicator

    if ucv is not None:
        try:
            ucv.disconnect()
        except Exception:
            pass
        ucv = None
        communicator = None

    ucv, communicator = geh.ensure_connection()
    return ucv, communicator

In [ ]:
ucv, communicator = ensure_connection()

In [ ]:
# ---- 実行パラメータ（Notebook 既定は小規模テスト） ----
GRID_N = 3  # 本番 10,000 箱: 100
SPAWN_INTERVAL_S = 0.005
CUBE_ENABLE_PHYSICS = True
CUBE_SPAWN_ABOVE_FLOOR_CM = 5.0  # 大きいと角の箱が床外へ弾ける
AGENT_ENABLE_PHYSICS = False  # True でも Humanoid/Robot では無視（ラグドール防止）

os.environ["GRID_N"] = str(GRID_N)
os.environ["SPAWN_INTERVAL_S"] = str(SPAWN_INTERVAL_S)
os.environ["CUBE_ENABLE_PHYSICS"] = "1" if CUBE_ENABLE_PHYSICS else "0"
os.environ["CUBE_SPAWN_ABOVE_FLOOR_CM"] = str(CUBE_SPAWN_ABOVE_FLOOR_CM)
os.environ["AGENT_ENABLE_PHYSICS"] = "1" if AGENT_ENABLE_PHYSICS else "0"

import grid_env_hri_simulation as geh

importlib.reload(geh)

cube_registry: dict = {}
human_name: Optional[str] = None
robot_ok: bool = False

print(
    f"[Config] GRID_N={geh.GRID_N} ({geh.GRID_N ** 2} cubes), "
    f"floor_top_z={geh.FLOOR_TOP_Z_CM} cm, origin={geh.MAP_ORIGIN_XY_CM}"
)
print(
    f"  human map={geh.HUMAN_MAP_XY_M} m, robot map={geh.ROBOT_MAP_XY_M} m, "
    f"cube_physics={geh.CUBE_ENABLE_PHYSICS}, cube_drop_cm={geh.CUBE_SPAWN_ABOVE_FLOOR_CM}, "
    f"human_z={geh.HUMAN_SPAWN_Z_CM}, robot_z={geh.ROBOT_SPAWN_Z_CM}"
)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, ax = plt.subplots(figsize=(6, 6))
floor_m = geh.FLOOR_SIZE_M
ax.add_patch(
    mpatches.Rectangle((0, 0), floor_m, floor_m, fill=False, edgecolor="black", linewidth=2)
)

n = geh.GRID_N
cube_m = geh.CUBE_SIZE_M
for row in range(n):
    for col in range(n):
        ax.add_patch(
            mpatches.Rectangle(
                (col * cube_m, row * cube_m),
                cube_m,
                cube_m,
                facecolor="#4c72b0",
                alpha=0.25,
                edgecolor="#4c72b0",
                linewidth=0.3,
            )
        )

hx, hy = geh.HUMAN_MAP_XY_M
rx, ry = geh.ROBOT_MAP_XY_M
ax.plot(hx, hy, "o", color="tab:orange", markersize=10, label="Humanoid")
ax.plot(rx, ry, "s", color="tab:green", markersize=10, label="SpotDog")

for idx, (mx, my) in enumerate(geh.DEMO_SOLID_MAP_XY_M):
    ax.plot(mx, my, "s", color="dimgray", markersize=8, label="Demo solid" if idx == 0 else None)
for idx, (mx, my) in enumerate(geh.DEMO_TRANSLUCENT_MAP_XY_M):
    ax.plot(mx, my, "D", color="silver", markersize=9, label="Demo translucent" if idx == 0 else None)
    ax.add_patch(
        mpatches.Rectangle(
            (mx - 0.5, my - 0.5),
            1.0,
            1.0,
            fill=False,
            edgecolor="crimson",
            linewidth=1.5,
            linestyle="--",
        )
    )

ax.set_xlim(-0.5, floor_m + 0.5)
ax.set_ylim(-0.5, floor_m + 0.5)
ax.set_aspect("equal")
ax.set_xlabel("x [m] (map, lower-left origin)")
ax.set_ylabel("y [m]")
ax.set_title(f"Spawn layout preview (GRID_N={n}, {n * n} cubes)")
ax.legend(loc="upper right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
if not geh.spawn_fixed_floor(ucv):
    raise RuntimeError("[Floor] spawn failed — PAK / BP パス / SimWorld 再起動を確認")

In [ ]:
cube_registry = geh.spawn_cubes(ucv, geh.GRID_N)
print(f"[Cubes] registry size: {len(cube_registry)}")

In [ ]:
marker_registry: dict = {}
if geh.SPAWN_DEMO_MODE_CUBES:
    marker_registry = geh.spawn_demo_mode_cubes(ucv)
print(f"[DemoCubes] registry size: {len(marker_registry)}")

In [ ]:
human_name = geh.spawn_humanoid(communicator, ucv)
robot_ok = geh.spawn_robot(ucv)
print(f"[Agents] humanoid={human_name}, robot_ok={robot_ok}")

In [ ]:
geh.settle_after_cube_spawn_if_needed()

geh.report_spawn_state(
    ucv, cube_registry, human_name, marker_registry=marker_registry or None
)

print("[Done]")
print(f"  floor: {geh.FLOOR_ACTOR_NAME}")
print(f"  cubes: {len(cube_registry)}")
print(f"  humanoid: {human_name}")
print(f"  robot: {geh.ROBOT_ACTOR_NAME if robot_ok else 'FAILED'}")

## SpotDog 通過試験（ログ判定）

### A. 単一オブジェクトの ON/OFF 切替のみ（推奨・シンプル）

立方体 **1 個**（既定: `toggle_test_cube` @ 5.5, 5.5 m）に対し:

| フェーズ | SetBlocking | Robot 期待 |
|----------|-------------|------------|
| OFF | False | 通過（PASS） |
| ON | True | 非通過（BLOCK） |
| OFF | False | 再通過（PASS） |

```python
if not robot_ok:
    raise RuntimeError("[PassageTest] SpotDog not spawned")
toggle_ok = geh.run_single_cube_toggle_passage_suite(ucv)
if not toggle_ok:
    raise RuntimeError("[PassageTest] single-cube toggle FAILED")
```

CLI: `python run_single_cube_toggle_test.py`（床+1箱+Robot のみ）

### B. 複数デモ立方体（L字3 + 透過1 + 切替）

`run_all_demo_passage_tests(ucv, marker_registry)` — 固定4件 + `demo_translucent_00` 切替3件。


In [ ]:
# True = 立方体1個の OFF/ON/OFF 切替試験のみ / False = 複数デモ立方体の一括試験
RUN_SINGLE_CUBE_TOGGLE_ONLY = True

if not robot_ok:
    raise RuntimeError("[PassageTest] SpotDog not spawned — run agents cell first")

if RUN_SINGLE_CUBE_TOGGLE_ONLY:
    toggle_ok = geh.run_single_cube_toggle_passage_suite(ucv)
    if not toggle_ok:
        raise RuntimeError("[PassageTest] single-cube OFF/ON toggle FAILED")
else:
    if not marker_registry:
        print("[PassageTest] skip: no demo cubes (SPAWN_DEMO_MODE_CUBES=0?)")
    else:
        passage_all_ok = geh.run_all_demo_passage_tests(ucv, marker_registry)
        if not passage_all_ok:
            raise RuntimeError("[PassageTest] FAILED — see [PassageTest] logs above")


## クリーンアップ（任意）

スポーンした床・箱・ロボットを Unreal から削除します。Humanoid は Communicator 経由のため、必要に応じて UE 側で手動削除してください。

In [ ]:
# RUN_CLEANUP = True にしてから実行
RUN_CLEANUP = True

if RUN_CLEANUP:
    geh.cleanup_spawned(
        ucv,
        cube_registry.keys(),
        marker_ids=marker_registry.keys(),
    )
    if human_name:
        geh.destroy_if_exists(ucv, human_name)
    print("[Cleanup] floor, transparent cubes, demo cubes, robot destroyed.")
else:
    print("[Cleanup] skipped (set RUN_CLEANUP = True to destroy spawned actors)")